# Phase 5 — 4-Channel Graph RAG Pipeline

**Retrieval channels:**
1. **Qwen dense semantic** — OpenAI-compatible embeddings → Neo4j vector index
2. **ColBERT semantic** — `lightonai/Reason-ModernColBERT` MaxSim over `.npy` files
3. **Lucene BM25** — fulltext index in the `colbert` Neo4j database
4. **Qwen KG** — entity extraction → graph traversal in the `colbert` Neo4j database

All four channels feed into **RRF fusion → Cohere reranker → Qwen generation**.

In [1]:
! pip install langfuse


(rag) k:\Ds\UCD\CeADAR_Quan\research\FinalizedPipelines>doskey neo=ssh -i C:\Users\kiree\.ssh\id_ed25519 -L 7474:127.0.0.1:7474 -L 7687:127.0.0.1:7687 krishna@35.186.40.29 


In [2]:
from __future__ import annotations

import json
import logging
import os
import pickle
import re
import sys
import time
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

# ── path so colbert_retrieval can be imported from research/1 ────────────────
_HERE = Path(__file__).parent if "__file__" in dir() else Path(".").resolve()
_R1   = (_HERE / "../1").resolve()
if str(_R1) not in sys.path:
    sys.path.insert(0, str(_R1))

# Suppress Pydantic serialization warnings from LangChain structured output
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

import numpy as np
from dotenv import load_dotenv
from neo4j import GraphDatabase
from pydantic import BaseModel, Field

# ColBERT — reuses encode() and maxsim_score() from research/1
from colbert_retrieval import encode as colbert_encode, maxsim_score

# ── LangChain ────────────────────────────────────────────────────────────────
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import OpenAIEmbeddings

try:
    from langchain_ollama import ChatOllama
    HAS_OLLAMA = True
except ImportError:
    HAS_OLLAMA = False


try:
    from langchain_huggingface import HuggingFaceEmbeddings
    HAS_HF = True
except ImportError:
    HAS_HF = False

# Langfuse tracing
from contextvars import copy_context

_LANGFUSE_AVAILABLE = False
langfuse_context = None
def observe(**kwargs):
    return lambda f: f
def _get_langfuse_client():
    return None

try:
    from langfuse.decorators import observe, langfuse_context
    _LANGFUSE_AVAILABLE = True
except ImportError:
    pass

# Client accessor — works with both Langfuse v2 (Langfuse()) and v3 (get_client())
if _LANGFUSE_AVAILABLE:
    try:
        from langfuse import get_client as _get_langfuse_client      # v3
    except ImportError:
        from langfuse import Langfuse as _LangfuseV2
        _langfuse_v2_singleton = None
        def _get_langfuse_client():
            global _langfuse_v2_singleton
            if _langfuse_v2_singleton is None:
                _langfuse_v2_singleton = _LangfuseV2()
            return _langfuse_v2_singleton


In [3]:
logger = logging.getLogger("graphrag")

LUCENE_SPECIAL = re.compile(r'([+\-&|!(){}\[\]^"~*?:\\/])')

VIETNAMESE_PATTERN = re.compile(
    r"[\u00e0\u00e1\u1ea3\u00e3\u1ea1\u0103\u1eaf\u1eb1\u1eb3\u1eb5\u1eb7"
    r"\u00e2\u1ea5\u1ea7\u1ea9\u1eab\u1ead\u00e8\u00e9\u1ebb\u1ebd\u1eb9"
    r"\u00ea\u1ebf\u1ec1\u1ec3\u1ec5\u1ec7\u00ec\u00ed\u1ec9\u0129\u1ecb"
    r"\u00f2\u00f3\u1ecf\u00f5\u1ecd\u00f4\u1ed1\u1ed3\u1ed5\u1ed7\u1ed9"
    r"\u01a1\u1edb\u1edd\u1edf\u1ee1\u1ee3\u00f9\u00fa\u1ee7\u0169\u1ee5"
    r"\u01b0\u1ee9\u1eeb\u1eed\u1eef\u1ef1\u1ef3\u00fd\u1ef7\u1ef9\u1ef5"
    r"\u0111\u0110]"
)

MAX_QUERY_LENGTH = 2000


In [4]:
@dataclass
class RAGConfig:
    """Central configuration for the 4-channel Graph RAG pipeline."""

    # Neo4j — KG + Lucene BM25 live here (colbert database, no vector index)
    neo4j_uri: str = "bolt://localhost:7687"
    neo4j_user: str = "neo4j"
    neo4j_password: str = ""
    neo4j_database: str = "neo4j"   # vector index + fulltext + KG

    # Search params
    vector_search_k: int = 5
    keyword_search_k: int = 3
    colbert_search_k: int = 10
    graph_traversal_top_k: int = 20
    max_graph_hops: int = 1
    rrf_k: int = 60

    # Mode
    mode: str = "textonly"

    # LLM
    llm_provider: str = "qwen"
    llm_model: str = "cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit"
    llm_temperature: float = 0.0

    # Dense embeddings (Qwen semantic channel — must match vector index dim)
    embed_provider: str = "openai"
    embed_model: str = "text-embedding-3-small"

    # ColBERT embeddings
    colbert_embed_base: str = str(Path("../1/colbert-embeddings").resolve())


    # Context budget
    max_context_tokens: int = 16000
    include_full_text: bool = True

    # Sibling expansion
    enable_sibling_expansion: bool = False

    # Image captions
    include_image_captions: bool = False

    # Farmer-friendly output
    verify_math: bool = True

    @classmethod
    def from_env(cls) -> RAGConfig:
        load_dotenv((_HERE / "../1/.env").resolve())
        return cls(
            neo4j_uri=os.getenv("NEO4J_URI", "bolt://localhost:7687"),
            neo4j_user=os.getenv("NEO4J_USER", "neo4j"),
            neo4j_password=os.getenv("NEO4J_PASSWORD", ""),
            neo4j_database=os.getenv("NEO4J_DATABASE", "neo4j"),
            vector_search_k=int(os.getenv("VECTOR_SEARCH_K", "5")),
            keyword_search_k=int(os.getenv("KEYWORD_SEARCH_K", "3")),
            colbert_search_k=int(os.getenv("COLBERT_SEARCH_K", "10")),
            graph_traversal_top_k=int(os.getenv("GRAPH_TRAVERSAL_TOP_K", "20")),
            max_graph_hops=int(os.getenv("MAX_GRAPH_HOPS", "1")),
            llm_provider=os.getenv("LLM_PROVIDER", "qwen"),
            llm_model=os.getenv("QWEN_MODEL_NAME", "cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit"),
            llm_temperature=float(os.getenv("LLM_TEMPERATURE", "0.0")),
            embed_provider=os.getenv("EMBED_PROVIDER", "openai"),
            embed_model=os.getenv("EMBED_MODEL", "text-embedding-3-small"),
            colbert_embed_base=os.getenv(
                "COLBERT_EMBED_BASE",
                str((_HERE / "../1/colbert-embeddings").resolve()),
            ),
            mode=os.getenv("RAG_MODE", "textonly"),
            include_image_captions=os.getenv("INCLUDE_IMAGE_CAPTIONS", "false").lower() == "true",
            verify_math=os.getenv("VERIFY_MATH", "true").lower() == "true",
        )


In [5]:
def create_llm(config: RAGConfig) -> BaseChatModel:
    if config.llm_provider == "ollama":
        if not HAS_OLLAMA:
            raise ImportError("pip install langchain-ollama")
        return ChatOllama(model=config.llm_model, temperature=config.llm_temperature)
    elif config.llm_provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=config.llm_model, temperature=config.llm_temperature)
    elif config.llm_provider == "qwen":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(
            model=config.llm_model,
            base_url=os.getenv("QWEN_BASE_URL"),
            api_key=os.getenv("QWEN_API_KEY"),
            temperature=config.llm_temperature,
        )
    elif config.llm_provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=config.llm_model, temperature=config.llm_temperature)
    else:
        raise ValueError(f"Unknown LLM provider: {config.llm_provider}")


def create_embeddings(config: RAGConfig) -> Embeddings:
    if config.embed_provider == "openai":
        return OpenAIEmbeddings(model=config.embed_model)
    elif config.embed_provider == "huggingface":
        if not HAS_HF:
            raise ImportError("pip install langchain-huggingface")
        return HuggingFaceEmbeddings(model_name=config.embed_model)
    else:
        raise ValueError(f"Unknown embed provider: {config.embed_provider}")



In [6]:
def detect_language(text: str) -> str:
    return "vi" if len(VIETNAMESE_PATTERN.findall(text)) >= 2 else "en"


def translate_to_english(text: str, llm: BaseChatModel) -> str:
    response = llm.invoke([
        SystemMessage(content="You are a translator. Translate the following Vietnamese text to English. Return only the translation."),
        HumanMessage(content=text),
    ])
    return response.content.strip()


def sanitize_lucene(query: str) -> str:
    return LUCENE_SPECIAL.sub(r"\\\1", query)


In [7]:
class ChunkStore:
    """In-memory index of phase1_chunks_vi.json, keyed by chunk_id."""

    def __init__(self, json_path: str):
        logger.info("Loading chunk store from %s", json_path)
        with open(json_path, "r", encoding="utf-8") as f:
            chunks = json.load(f)
        self._by_id: dict[str, dict] = {str(c["chunk_id"]): c for c in chunks}
        # Report caption coverage
        with_captions = sum(1 for c in chunks if c.get("image_captions"))
        total_images = sum(len(c.get("images", [])) for c in chunks)
        total_captions = sum(len(c.get("image_captions", [])) for c in chunks)
        logger.info("Loaded %d chunks (%d images, %d captions across %d chunks)",
                    len(self._by_id), total_images, total_captions, with_captions)

    def get(self, chunk_id: str) -> dict | None:
        return self._by_id.get(str(chunk_id))

    def __len__(self) -> int:
        return len(self._by_id)


In [8]:
class ColBERTStore:
    """Loads per-chunk ColBERT .npy files into memory and runs MaxSim retrieval.

    Directory layout expected (produced by phase4_load_neo4j_colbert.py):
        <embed_base>/chunks_en/{chunk_id}.npy   — English summary embeddings
        <embed_base>/chunks_vi/{chunk_id}.npy   — Vietnamese summary embeddings
        <embed_base>/chunks_en/ids.pkl          — ordered list of chunk_ids
        <embed_base>/chunks_vi/ids.pkl
    Each .npy file has shape (num_tokens, 128), float32, L2-normalised.
    """

    def __init__(self, embed_base: str):
        logger.info("Loading ColBERT store from %s", embed_base)
        self._en = self._load_index(embed_base, "chunks_en")
        self._vi = self._load_index(embed_base, "chunks_vi")
        logger.info("ColBERT store ready — %d EN embeddings, %d VI embeddings",
                    len(self._en), len(self._vi))

    def _load_index(self, embed_base: str, index_name: str) -> dict[str, np.ndarray]:
        index_dir = os.path.join(embed_base, index_name)
        ids_file  = os.path.join(index_dir, "ids.pkl")
        if not os.path.exists(ids_file):
            logger.warning("ColBERT index not found: %s — channel will return empty", ids_file)
            return {}
        with open(ids_file, "rb") as f:
            ids = pickle.load(f)
        embeddings: dict[str, np.ndarray] = {}
        missing = 0
        for doc_id in ids:
            path = os.path.join(index_dir, f"{doc_id}.npy")
            if os.path.exists(path):
                embeddings[str(doc_id)] = np.load(path)
            else:
                missing += 1
        if missing:
            logger.warning("ColBERT index '%s': %d/%d .npy files missing", index_name, missing, len(ids))
        return embeddings

    def search(self, query_emb: np.ndarray, lang: str, k: int) -> list[tuple[str, float]]:
        """Return top-k (chunk_id, score) pairs using MaxSim scoring."""
        store = self._vi if lang == "vi" else self._en
        if not store:
            return []
        scored = [
            (doc_id, maxsim_score(query_emb, doc_emb))
            for doc_id, doc_emb in store.items()
        ]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]


In [9]:
class QueryEntities(BaseModel):
    entities: list[str] = Field(description="Entity names mentioned or implied in the query")


@dataclass
class ChunkTrace:
    chunk_id: str
    title: str
    hierarchy: str
    start_page: int
    end_page: int
    channels: list[str]
    rrf_score: float


@dataclass
class RAGResult:
    answer: str
    language: str
    traces: list[ChunkTrace]
    graph_triples: list[dict]
    community_summaries: list[str]
    timings: dict[str, float]
    config_snapshot: dict
    referenced_figures: list[dict] = field(default_factory=list)
    raw_answer: str = ""


In [10]:
def reciprocal_rank_fusion(
    ranked_lists: dict[str, list[tuple[str, float]]],
    k: int = 60,
) -> tuple[list[tuple[str, float]], dict[str, set[str]]]:
    """Fuse N ranked lists via RRF. Returns (sorted_results, channel_map)."""
    scores: dict[str, float] = defaultdict(float)
    channels: dict[str, set[str]] = defaultdict(set)
    for channel, ranked_list in ranked_lists.items():
        for rank, (chunk_id, _) in enumerate(ranked_list, start=1):
            scores[chunk_id] += 1.0 / (k + rank)
            channels[chunk_id].add(channel)
    sorted_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results, dict(channels)


In [11]:
_JSON_BLOCK_RE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)


def _extract_json(text: str) -> str:
    m = _JSON_BLOCK_RE.search(text)
    if m:
        return m.group(1).strip()
    for sc, ec in [("{", "}"), ("[", "]")]:
        s, e = text.find(sc), text.rfind(ec)
        if s != -1 and e != -1 and e > s:
            return text[s: e + 1]
    return text.strip()


def _parse_json_entity_list(text: str) -> list[str]:
    try:
        data = json.loads(_extract_json(text))
        if isinstance(data, dict) and "entities" in data:
            return [str(e) for e in data["entities"]]
        if isinstance(data, list):
            return [str(e) for e in data]
    except Exception:
        pass
    return []


In [12]:
# Figure-reference pattern: matches [Figure C24-1], [Figure C24-2], etc.
FIGURE_REF_RE = re.compile(r"\[Figure\s+(C\d+-\d+)\]", re.IGNORECASE)


class GraphRAGPipeline:
    """4-channel Graph RAG pipeline: ColBERT + Qwen dense + Lucene BM25 + Qwen KG."""

    def __init__(self, config: RAGConfig, chunks_json_path: str):
        self.config = config
        self.chunk_store  = ChunkStore(chunks_json_path)
        self.colbert_store = ColBERTStore(config.colbert_embed_base)
        self.driver = GraphDatabase.driver(
            config.neo4j_uri, auth=(config.neo4j_user, config.neo4j_password)
        )
        self.llm        = create_llm(config)
        self.embeddings = create_embeddings(config)
        self._validate_embedding_dimension()
        logger.info(
            "GraphRAGPipeline ready (mode=%s, captions=%s, llm=%s, "
            "db=%s, colbert_en=%d chunks, verify_math=%s)",
            config.mode, config.include_image_captions, config.llm_model,
            config.neo4j_database, len(self.colbert_store._en), config.verify_math,
        )

    # ── Startup validation ────────────────────────────────────────────────────

    def _validate_embedding_dimension(self):
        test_vec   = self.embeddings.embed_query("test")
        actual_dim = len(test_vec)
        expected_dim = 1536
        try:
            with self.driver.session(database=self.config.neo4j_database) as session:
                rec = session.run(
                    "SHOW INDEXES YIELD name, options "
                    "WHERE name = 'chunk_embedding_index' RETURN options"
                ).single()
                if rec:
                    expected_dim = rec["options"].get("indexConfig", {}).get("vector.dimensions", 1536)
        except Exception as e:
            logger.warning("Could not query vector index dimension: %s", e)
        if actual_dim != expected_dim:
            raise ValueError(
                f"Embedding dimension mismatch: model produces {actual_dim}d "
                f"but Neo4j index expects {expected_dim}d."
            )
        logger.info("Embedding dimension validated: %dd", actual_dim)

    # ── Neo4j helper ──────────────────────────────────────────────────────────

    def _run_cypher(self, query: str, params: dict | None = None,
                    database: str | None = None) -> list[dict]:
        db = database or self.config.neo4j_database
        with self.driver.session(database=db) as session:
            return [r.data() for r in session.run(query, params or {})]

    # ── Channel 1: Qwen dense semantic (vector index) ─────────────────────────

    @observe(name="semantic_search", capture_input=False, capture_output=False)
    def _semantic_search(self, query: str, lang: str,
                         k: int | None = None) -> list[tuple[str, float]]:
        k = k or self.config.vector_search_k
        if langfuse_context:
            langfuse_context.update_current_observation(
                input={"query": query, "lang": lang, "k": k},
                metadata={"channel": "qwen_dense_semantic"},
            )
        embedding   = self.embeddings.embed_query(query)
        index_name  = "chunk_vi_embedding_index" if lang == "vi" else "chunk_embedding_index"
        try:
            records = self._run_cypher(
                f"CALL db.index.vector.queryNodes('{index_name}', $k, $embedding) "
                "YIELD node, score RETURN node.chunk_id AS chunk_id, score",
                {"k": k, "embedding": embedding},
            )
            results = [(str(r["chunk_id"]), float(r["score"])) for r in records]
            if langfuse_context:
                langfuse_context.update_current_observation(
                    output={"count": len(results), "top_chunks": [r[0] for r in results[:3]]},
                    metadata={"channel": "qwen_dense_semantic", "index": index_name},
                )
            return results
        except Exception as e:
            logger.warning("Dense semantic search failed (%s): %s", index_name, e)
            if langfuse_context:
                langfuse_context.update_current_observation(output={"count": 0, "error": str(e)})
            return []

    # ── Channel 2: ColBERT semantic (numpy MaxSim) ────────────────────────────

    @observe(name="colbert_search", capture_input=False, capture_output=False)
    def _colbert_search(self, query: str, lang: str,
                        k: int | None = None) -> list[tuple[str, float]]:
        k = k or self.config.colbert_search_k
        if langfuse_context:
            langfuse_context.update_current_observation(
                input={"query": query, "lang": lang, "k": k},
                metadata={"channel": "colbert_semantic"},
            )
        try:
            query_emb = colbert_encode([query], is_query=True)[0]   # (Q, 128)
            results = self.colbert_store.search(query_emb, lang, k)
            if langfuse_context:
                langfuse_context.update_current_observation(
                    output={"count": len(results), "top_chunks": [r[0] for r in results[:3]]},
                    metadata={"channel": "colbert_semantic"},
                )
            return results
        except Exception as e:
            logger.warning("ColBERT search failed: %s", e)
            if langfuse_context:
                langfuse_context.update_current_observation(output={"count": 0, "error": str(e)})
            return []

    # ── Channel 3: Lucene BM25 fulltext ───────────────────────────────────────

    @observe(name="fulltext_search", capture_input=False, capture_output=False)
    def _fulltext_search(self, query: str, k: int | None = None) -> list[tuple[str, float]]:
        k = k or self.config.keyword_search_k
        if langfuse_context:
            langfuse_context.update_current_observation(
                input={"query": query, "k": k},
                metadata={"channel": "lucene_bm25"},
            )
        sanitized = sanitize_lucene(query)
        if not sanitized.strip():
            return []
        try:
            records = self._run_cypher(
                "CALL db.index.fulltext.queryNodes('chunk_fulltext_index', $query) "
                "YIELD node, score RETURN node.chunk_id AS chunk_id, score LIMIT $k",
                {"query": sanitized, "k": k},
                # uses self.config.neo4j_database (colbert db) by default
            )
            results = [(str(r["chunk_id"]), float(r["score"])) for r in records]
            if langfuse_context:
                langfuse_context.update_current_observation(
                    output={"count": len(results), "top_chunks": [r[0] for r in results[:3]]},
                    metadata={"channel": "lucene_bm25"},
                )
            return results
        except Exception as e:
            logger.warning("Fulltext search failed: %s", e)
            if langfuse_context:
                langfuse_context.update_current_observation(output={"count": 0, "error": str(e)})
            return []

    # ── Channel 4: Qwen KG graph retrieval ────────────────────────────────────

    @observe(name="graph_retrieval", capture_input=False, capture_output=False)
    def _graph_retrieval(self, query_en: str, query_embedding: list[float],
                         community_embedding: list[float] | None = None
    ) -> tuple[list[tuple[str, float]], list[dict], list[dict]]:
        if langfuse_context:
            langfuse_context.update_current_observation(
                input={"query": query_en},
                metadata={"channel": "qwen_kg_graph"},
            )
        entities = self._extract_entities(query_en)
        if not entities:
            return [], [], []

        sanitized_names = [sanitize_lucene(e) for e in entities if sanitize_lucene(e).strip()]
        if not sanitized_names:
            return [], [], []

        try:
            matched = self._run_cypher(
                "UNWIND $names AS name "
                "CALL db.index.fulltext.queryNodes('entity_fulltext_index', name) "
                "YIELD node, score WHERE score > 0.5 "
                "RETURN node.name AS name, node.type AS type, "
                "node.description AS description, score "
                "ORDER BY score DESC LIMIT $limit",
                {"names": sanitized_names, "limit": len(sanitized_names) * 5},
            )
        except Exception as e:
            logger.warning("Entity fulltext search failed: %s", e)
            matched = []

        if not matched:
            return [], [], []

        seen: set[str] = set()
        unique_entities = [e for e in matched if not (e["name"] in seen or seen.add(e["name"]))]
        entity_names = [e["name"] for e in unique_entities]

        all_triples: list[dict] = []
        chunk_entity_counts: dict[str, int] = defaultdict(int)

        try:
            records = self._run_cypher(
                "UNWIND $names AS name "
                "MATCH (e:Entity {name: name})-[r:RELATES_TO]-(n:Entity) "
                "RETURN "
                "  CASE WHEN startNode(r) = e THEN e.name ELSE n.name END AS source, "
                "  r.relation AS relation, "
                "  CASE WHEN startNode(r) = e THEN n.name ELSE e.name END AS target, "
                "  r.detail AS detail, r.source_chunk AS source_chunk, r.pages AS pages",
                {"names": entity_names},
            )
            all_triples.extend(records)
        except Exception as e:
            logger.warning("Graph traversal failed: %s", e)

        try:
            records = self._run_cypher(
                "UNWIND $names AS name "
                "MATCH (e:Entity {name: name})-[:EXTRACTED_FROM]->(c:Chunk) "
                "RETURN c.chunk_id AS chunk_id",
                {"names": entity_names},
            )
            for r in records:
                chunk_entity_counts[str(r["chunk_id"])] += 1
        except Exception as e:
            logger.warning("Provenance query failed: %s", e)

        for t in all_triples:
            sc = t.get("source_chunk")
            if sc:
                chunk_entity_counts[str(sc)] += 1

        ranked = sorted(chunk_entity_counts.items(), key=lambda x: x[1], reverse=True)
        max_count = ranked[0][1] if ranked else 1
        chunk_results = [(cid, cnt / max_count) for cid, cnt in ranked][: self.config.graph_traversal_top_k]

        communities: list[dict] = []
        try:
            # Community search via vector index — reuse embedding from query() to avoid extra API call
            comm_emb = community_embedding if community_embedding is not None else query_embedding
            records = self._run_cypher(
                "CALL db.index.vector.queryNodes('community_embedding_index', 3, $embedding) "
                "YIELD node, score WHERE score > 0.3 "
                "RETURN node.title AS title, node.summary AS summary, score",
                {"embedding": comm_emb},
            )
            communities = records
        except Exception as e:
            logger.warning("Community search failed: %s", e)

        seen_t: set[str] = set()
        unique_triples = []
        for t in all_triples:
            key = f"{t.get('source')}|{t.get('relation')}|{t.get('target')}"
            if key not in seen_t:
                seen_t.add(key)
                unique_triples.append(t)

        if langfuse_context:
            langfuse_context.update_current_observation(
                output={
                    "chunk_count": len(chunk_results),
                    "triple_count": len(unique_triples),
                    "community_count": len(communities),
                    "entity_count": len(entity_names),
                },
                metadata={"channel": "qwen_kg_graph"},
            )
        return chunk_results, unique_triples, communities

    @observe(name="entity_extraction", capture_input=False, capture_output=False)
    def _extract_entities(self, query_en: str) -> list[str]:
        if langfuse_context:
            langfuse_context.update_current_observation(input={"query": query_en})
        prompt = (
            "Extract the key entity names from this query that would be useful "
            "for searching a knowledge graph about rice production. "
            "Include equipment, processes, inputs, crop stages, pests, parameters, "
            "products, principles, and varieties.\n\n"
            f"Query: {query_en}"
        )
        try:
            result = self.llm.with_structured_output(QueryEntities).invoke(
                [HumanMessage(content=prompt)]
            )
            if result and result.entities:
                if langfuse_context:
                    langfuse_context.update_current_observation(output={"entities": result.entities})
                return result.entities
        except Exception as e:
            logger.warning("Entity extraction failed: %s", e)
        if langfuse_context:
            langfuse_context.update_current_observation(output={"entities": []})
        return []


    # ── Sibling expansion ─────────────────────────────────────────────────────

    def _expand_siblings(self, chunk_ids: list[str]) -> list[str]:
        if not self.config.enable_sibling_expansion:
            return chunk_ids
        expanded = list(chunk_ids)
        for cid in chunk_ids:
            try:
                records = self._run_cypher(
                    "MATCH (parent)-[:CONTAINS]->(target:Chunk {chunk_id: $chunk_id}) "
                    "MATCH (parent)-[:CONTAINS]->(sibling:Chunk) "
                    "WHERE sibling.chunk_id <> $chunk_id "
                    "RETURN sibling.chunk_id AS chunk_id",
                    {"chunk_id": cid},
                )
                for r in records:
                    sib = str(r["chunk_id"])
                    if sib not in expanded:
                        expanded.append(sib)
            except Exception as e:
                logger.warning("Sibling expansion failed for %s: %s", cid, e)
        return expanded

    # ── Context assembly ──────────────────────────────────────────────────────

    def _assemble_context(
        self,
        chunk_ids: list[str],
        lang: str,
        mode: str,
        triples: list[dict],
        communities: list[dict],
    ) -> tuple[str, list[dict], dict[str, dict]]:
        """Returns (context_text, image_blocks, figure_map)."""
        budget = self.config.max_context_tokens
        used   = 0
        context_parts: list[str] = []
        image_blocks:  list[dict] = []
        figure_map:    dict[str, dict] = {}

        def tok(t): return len(t) // 3

        if triples:
            lines = []
            for t in triples[:30]:
                line = f"  {t.get('source','?')} --[{t.get('relation','?')}]--> {t.get('target','?')}"
                if t.get("detail"):
                    line += f": {t['detail']}"
                lines.append(line)
            g = "=== Knowledge Graph Triples ===\n" + "\n".join(lines)
            used += tok(g); context_parts.append(g)

        if communities:
            lines = [f"  [{c.get('title','')}]: {c.get('summary','')}" for c in communities[:5]]
            cm = "=== Community Summaries ===\n" + "\n".join(lines)
            used += tok(cm); context_parts.append(cm)

        tk  = "eng_chunk_title"     if lang == "en" else "vi_chunk_title"
        hk  = "eng_chunk_hierarchy" if lang == "en" else "hierarchy_id"
        sk  = "eng_summary"         if lang == "en" else "vi_summary"
        txk = "eng_text"            if lang == "en" else "vi_text"

        for cid in chunk_ids:
            chunk = self.chunk_store.get(cid)
            if chunk is None:
                continue

            title    = chunk.get(tk,  f"Chunk {cid}")
            hier     = chunk.get(hk,  "")
            summary  = chunk.get(sk,  "")
            fulltext = chunk.get(txk, "")
            tables   = chunk.get("tables", [])
            images   = chunk.get("images", [])
            captions = chunk.get("image_captions", [])
            pages    = f"(pp. {chunk.get('start_page','?')}-{chunk.get('end_page','?')})"

            base = f"\n--- {title} {pages} ---\n[{hier}]\n{summary}"
            if used + tok(base) > budget:
                break
            used += tok(base)
            text = base

            ft = f"\n[Full text]\n{fulltext}" if self.config.include_full_text and fulltext else ""
            tb = "\n" + "\n".join(tables) if tables else ""

            if ft and used + tok(ft) <= budget:
                text += ft; used += tok(ft)
            if tb and used + tok(tb) <= budget:
                text += tb; used += tok(tb)

            if self.config.include_image_captions and captions:
                cap_lines = []
                for img_i, cap in enumerate(captions):
                    if not cap or cap.startswith("[CAPTION ERROR") or cap.startswith("[Empty"):
                        continue
                    fig_id = f"C{cid}-{img_i+1}"
                    cap_lines.append(f'  - [Figure {fig_id}]: "{cap}"')
                    figure_map[fig_id] = {"chunk_id": cid, "image_index": img_i,
                                          "caption": cap,
                                          "base64": images[img_i] if img_i < len(images) else None}
                if cap_lines:
                    cb = "\n[Figures in this section:]\n" + "\n".join(cap_lines)
                    if used + tok(cb) <= budget:
                        text += cb; used += tok(cb)

            context_parts.append(text)

            if mode == "multimodal" and images:
                for b64 in images:
                    if isinstance(b64, str) and b64:
                        image_blocks.append({"type": "image_url",
                                             "image_url": {"url": f"data:image/jpeg;base64,{b64}"}})

        return "\n".join(context_parts), image_blocks, figure_map

    # ── Answer generation ─────────────────────────────────────────────────────

    @observe(name="generate_answer", as_type="generation", capture_input=False, capture_output=False)
    def _generate_answer(self, query: str, context: str,
                         image_blocks: list[dict], lang: str, mode: str) -> str:
        if langfuse_context:
            langfuse_context.update_current_observation(
                model=self.config.llm_model,
                input={"query": query, "context_chars": len(context), "lang": lang, "mode": mode},
                metadata={"llm_provider": self.config.llm_provider, "lang": lang, "mode": mode},
            )
        if lang == "vi":
            sys_prompt = (
                "Ban la mot tro ly nong nghiep thuc te. Tra loi CHI dua tren ngu canh duoc cung cap.\n\n"
                "CAU TRUC TRA LOI (bat buoc theo thu tu):\n"
                "**Tra loi nhanh:** 1-2 cau trai loi thang van de.\n"
                "**Huong dan tung buoc:** Danh so ro rang, su dung ✅ cho moi buoc kha thi.\n"
                "**Luong/Don vi:** Neu bai viet dung don vi khac voi cau hoi, doi ra don vi quen thuoc "
                "(vi du: kg/ha → kg/cong/sao, lit/ha → binh xit 16 lit). Hien thi ro phep tinh.\n"
                "**Trich dan nguon:** Ghi so trang sau moi thong tin quan trong, vi du (tr. 45-47).\n"
                "**Canh bao:** Neu co thuoc tru sau/hoa chat, ghi ro ⚠️ lieu luong an toan va thoi gian cach ly.\n"
                "**Giai thich tu ngu:** Neu dung thuat ngu ky thuat, giai thich trong ngoac don, vi du "
                "'NH4+ (dan dau amoni — chua nito cho lua)'.\n"
                "**Neu khong chac:** Neu ngu canh chua du thong tin, ghi ro 'Tai lieu khong neu ro diem nay — "
                "nen hoi can bo khuyen nong dia phuong.'\n"
                "**Bai hoc chinh:** Mot dong tom tat hanh dong quan trong nhat.\n\n"
                "TU KIEM TRA (lam tham truoc khi viet cau tra loi cuoi cung):\n"
                "- Doc lai moi con so, ngay thang, phep doi don vi va phep tinh ban sap viet.\n"
                "- Doi chieu tung con so voi ngu canh nguon — neu nguon ghi 70 kg/ha, khong duoc viet 700 kg/ha.\n"
                "- Kiem tra tung buoc chuyen doi don vi (vi du: kg/ha sang kg/cong).\n"
                "- Neu phat hien sai, sua ngay tai cho truoc khi xuat ket qua.\n"
                "- KHONG de cap buoc tu kiem tra nay trong cau tra loi.\n\n"
                "Tra loi bang tieng Viet, van phong gian di, thuc te.\n\n"
                "HINH ANH: Neu ngu canh co muc '[Figures in this section:]', tham chieu hinh anh "
                "bang [Figure C<id>-<so>] neu thuc su giup minh hoa."
            )
        else:
            sys_prompt = (
                "You are a practical agricultural advisor helping rice farmers. "
                "Answer ONLY based on the provided context. "
                "Write as if explaining to a farmer who is standing in their field — clear, direct, and actionable.\n\n"
                "REQUIRED ANSWER STRUCTURE (follow this order):\n"
                "**Quick Answer:** 1-2 sentences directly addressing the question.\n"
                "**Step-by-Step Guide:** Numbered steps, each starting with ✅. Use plain language.\n"
                "**Amounts & Units:** If the source uses different units than the question implies, "
                "convert them to practical field amounts (e.g., kg/ha → kg per standard plot, "
                "L/ha → litres per backpack sprayer tank). Show the conversion clearly.\n"
                "**Sources:** After each key fact, cite the page range: (pp. 45-47).\n"
                "**Warnings:** For any pesticide, fertiliser rate, or chemical — prefix with ⚠️ "
                "and include safe handling notes and pre-harvest interval if mentioned.\n"
                "**Plain language:** If you must use a technical term, explain it in parentheses: "
                "e.g., 'tillering (the stage when the rice plant produces side shoots)'.\n"
                "**If context is thin:** Say clearly: 'The manual does not cover this specifically — "
                "please consult your local extension officer.'\n"
                "**Key Takeaway:** One bold sentence summarising the single most important action.\n\n"
                "SELF-CHECK (do this silently before writing your final answer):\n"
                "- Re-read every number, date, unit conversion, and calculation you are about to write.\n"
                "- Cross-check each figure against the source context — if the context says 70 kg/ha, do not write 700 kg/ha.\n"
                "- Verify all unit conversions step-by-step (e.g. kg/ha to kg per plot).\n"
                "- If you spot an inconsistency, fix it in place before outputting.\n"
                "- Do NOT mention this self-check in your answer.\n\n"
                "Answer in English. Be concise but complete — a farmer should be able to act on this immediately.\n\n"
                "FIGURES: The context may include a '[Figures in this section:]' block. "
                "If a figure genuinely helps, reference it inline as [Figure C<chunk_id>-<number>]. "
                "Only reference figures that are directly relevant."
            )

        full_prompt = f"{context}\n\nQuestion: {query}"
        if mode == "multimodal" and image_blocks:
            human_content: list[dict] = [{"type": "text", "text": full_prompt}]
            human_content.extend(image_blocks)
        else:
            human_content = full_prompt

        messages = [SystemMessage(content=sys_prompt), HumanMessage(content=human_content)]
        try:
            answer = self.llm.invoke(messages).content.strip()
            if langfuse_context:
                langfuse_context.update_current_observation(
                    output=answer[:500] if len(answer) > 500 else answer,
                    usage={"input": len(context) // 4, "output": len(answer) // 4},
                )
            return answer
        except Exception as e:
            logger.error("Answer generation failed: %s", e)
            if langfuse_context:
                langfuse_context.update_current_observation(output=f"[Generation failed: {e}]")
            return f"[Generation failed: {e}]"


    # ── Main query pipeline ───────────────────────────────────────────────────

    @observe(name="rag_pipeline", capture_input=False, capture_output=False)
    def query(self, user_query: str, mode: str | None = None) -> RAGResult:
        timings: dict[str, float] = {}
        mode = mode or self.config.mode

        # ── Langfuse: set root trace metadata ────────────────────────────────────
        _lf = _get_langfuse_client() if _LANGFUSE_AVAILABLE else None
        # Inline spans use start_as_current_observation (v3+ only); skip in v2
        if _lf and not hasattr(_lf, 'start_as_current_observation'):
            _lf = None
        if langfuse_context:
            langfuse_context.update_current_trace(
                input={"query": user_query, "mode": mode},
                metadata={"config": self._config_snapshot()},
                tags=["4-channel-rag", f"mode:{mode}"],
            )

        if not user_query or not user_query.strip():
            return RAGResult(answer="Please provide a non-empty query.", language="en",
                             traces=[], graph_triples=[], community_summaries=[],
                             timings={}, config_snapshot=self._config_snapshot())
        if len(user_query) > MAX_QUERY_LENGTH:
            user_query = user_query[:MAX_QUERY_LENGTH]

        # 1. Language detection
        t0 = time.time()
        if _lf:
            with _lf.start_as_current_observation(
                as_type="span", name="language_detection",
                input={"query": user_query[:300]},
            ) as _obs:
                lang = detect_language(user_query)
                _obs.update(output={"lang": lang})
        else:
            lang = detect_language(user_query)
        timings["language_detection"] = time.time() - t0
        logger.info("Query language: %s", lang)

        # 2. Translation
        t0 = time.time()
        if lang == "vi":
            if _lf:
                with _lf.start_as_current_observation(
                    as_type="span", name="translation",
                    input={"query_vi": user_query[:300]},
                ) as _obs:
                    query_en = translate_to_english(user_query, self.llm)
                    _obs.update(output={"query_en": query_en[:300]})
            else:
                query_en = translate_to_english(user_query, self.llm)
        else:
            query_en = user_query
        timings["translation"] = time.time() - t0

        # 3. Dense embedding (shared by semantic + community search)
        t0 = time.time()
        if _lf:
            with _lf.start_as_current_observation(
                as_type="span", name="query_embedding",
                input={"text": (user_query if lang == "en" else query_en)[:300]},
            ) as _obs:
                query_embedding = self.embeddings.embed_query(user_query if lang == "en" else query_en)
                _obs.update(
                    output={"embedding_dim": len(query_embedding)},
                    metadata={"model": self.config.embed_model},
                )
        else:
            query_embedding = self.embeddings.embed_query(user_query if lang == "en" else query_en)
        timings["embedding"] = time.time() - t0

        # 4. Four retrieval channels — run in parallel (all I/O-bound)
        # Each copy_context() call creates an independent context snapshot so
        # all 4 threads share the same Langfuse parent span but do not conflict.
        t_retrieval = time.time()
        with ThreadPoolExecutor(max_workers=4) as ex:
            f_semantic  = ex.submit(copy_context().run, self._semantic_search, user_query, lang)
            f_colbert   = ex.submit(copy_context().run, self._colbert_search, query_en, "en")
            f_fulltext  = ex.submit(copy_context().run, self._fulltext_search, user_query)
            f_graph     = ex.submit(
                copy_context().run, self._graph_retrieval, query_en, query_embedding, query_embedding
            )
            semantic_ranked            = f_semantic.result()
            colbert_ranked             = f_colbert.result()
            fulltext_ranked            = f_fulltext.result()
            graph_chunks, triples, communities = f_graph.result()

        timings["retrieval_parallel"] = time.time() - t_retrieval
        logger.info(
            "Retrieval counts — dense: %d, colbert: %d, bm25: %d, graph: %d  (parallel)",
            len(semantic_ranked), len(colbert_ranked), len(fulltext_ranked), len(graph_chunks),
        )

        # 5. RRF fusion across all 4 channels
        t0 = time.time()
        ranked_lists = {k: v for k, v in {
            "semantic_dense":   semantic_ranked,
            "semantic_colbert": colbert_ranked,
            "fulltext_bm25":    fulltext_ranked,
            "graph":            graph_chunks,
        }.items() if v}

        if not ranked_lists:
            return RAGResult(answer="No relevant information found.", language=lang,
                             traces=[], graph_triples=[], community_summaries=[],
                             timings=timings, config_snapshot=self._config_snapshot())

        fused, channel_map = reciprocal_rank_fusion(ranked_lists, k=self.config.rrf_k)
        timings["rrf_fusion"] = time.time() - t0

        # 6. Top-K selection by RRF score
        rrf_scores = dict(fused)
        top_ids   = [cid for cid, _ in fused[: self.config.graph_traversal_top_k]]

        # 7. Sibling expansion
        final_ids = self._expand_siblings(top_ids)

        # 8. Assemble context
        t0 = time.time()
        context, image_blocks, figure_map = self._assemble_context(
            final_ids, lang, mode, triples, communities
        )
        timings["context_assembly"] = time.time() - t0

        # 9. Generate answer
        t0 = time.time()
        answer = self._generate_answer(user_query, context, image_blocks, lang, mode)
        timings["generation"] = time.time() - t0

        # 10. Extract figure references from answer
        referenced_figures = []
        for m in FIGURE_REF_RE.finditer(answer):
            fig_id = m.group(1)
            if fig_id in figure_map:
                referenced_figures.append({
                    "figure_id":   fig_id,
                    "caption":     figure_map[fig_id]["caption"],
                    "chunk_id":    figure_map[fig_id]["chunk_id"],
                    "image_index": figure_map[fig_id]["image_index"],
                })
        if referenced_figures:
            logger.info("LLM referenced %d figure(s): %s",
                        len(referenced_figures), [f["figure_id"] for f in referenced_figures])

        # 11. Build traces
        traces = []
        for cid in final_ids:
            chunk = self.chunk_store.get(cid)
            if chunk is None:
                continue
            traces.append(ChunkTrace(
                chunk_id=cid,
                title=chunk.get("eng_chunk_title" if lang == "en" else "vi_chunk_title", f"Chunk {cid}"),
                hierarchy=chunk.get("eng_chunk_hierarchy" if lang == "en" else "hierarchy_id", ""),
                start_page=chunk.get("start_page", 0),
                end_page=chunk.get("end_page", 0),
                channels=sorted(channel_map.get(cid, set())),
                rrf_score=rrf_scores.get(cid, 0.0),
            ))

        timings["total"] = sum(timings.values())
        logger.info("Pipeline completed in %.2fs", timings["total"])

        # ── Langfuse: finalise trace with output + timing summary ─────────────────
        if langfuse_context:
            langfuse_context.update_current_trace(
                output={
                    "answer": answer[:500] if len(answer) > 500 else answer,
                    "language": lang,
                },
                metadata={
                    "timings": {k: round(v, 3) for k, v in timings.items()},
                    "chunks_retrieved": len(final_ids),
                    "triples_found": len(triples),
                    "communities_found": len(communities),
                    "channels_active": list(ranked_lists.keys()),
                },
                tags=["4-channel-rag", f"mode:{mode}", f"lang:{lang}"],
            )

        return RAGResult(
            answer=answer, language=lang, traces=traces,
            graph_triples=triples,
            community_summaries=[f"[{c.get('title','')}]: {c.get('summary','')}" for c in communities],
            timings=timings, config_snapshot=self._config_snapshot(),
            referenced_figures=referenced_figures,
            raw_answer=answer,
        )

    def _config_snapshot(self) -> dict:
        return {
            "llm_provider":            self.config.llm_provider,
            "llm_model":               self.config.llm_model,
            "embed_provider":          self.config.embed_provider,
            "embed_model":             self.config.embed_model,
            "neo4j_database":          self.config.neo4j_database,
            "colbert_embed_base":      self.config.colbert_embed_base,
            "vector_search_k":         self.config.vector_search_k,
            "colbert_search_k":        self.config.colbert_search_k,
            "keyword_search_k":        self.config.keyword_search_k,
            "graph_traversal_top_k":   self.config.graph_traversal_top_k,
            "rrf_k":                   self.config.rrf_k,
            "mode":                    self.config.mode,
            "max_context_tokens":      self.config.max_context_tokens,
            "include_full_text":       self.config.include_full_text,
            "enable_sibling_expansion":self.config.enable_sibling_expansion,
            "include_image_captions":  self.config.include_image_captions,
            "verify_math":             self.config.verify_math,
        }

    def close(self):
        self.driver.close()
        logger.info("Pipeline closed")


In [13]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("neo4j").setLevel(logging.WARNING)


In [14]:
# ── Runtime configuration ────────────────────────────────────────────────────
config = RAGConfig.from_env()

# ── Langfuse: verify connection now that .env vars are loaded ────────────────
if _LANGFUSE_AVAILABLE:
    try:
        _lf_check = _get_langfuse_client()
        if hasattr(_lf_check, "auth_check"):
            _lf_check.auth_check()
        _lf_host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
        print(f"Langfuse connected  : {_lf_host}")
    except Exception as _lf_err:
        print(f"Langfuse init warning (tracing will be skipped): {_lf_err}")
else:
    print("Langfuse not installed — run: pip install langfuse")

# Override for interactive use — remove / adjust as needed
config.llm_provider        = "qwen"
config.llm_model           = os.getenv("QWEN_MODEL_NAME", "cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit")
config.neo4j_database      = "neo4j"     # vector index + fulltext + KG
config.mode                = "textonly"

CHUNKS_JSON = str((_HERE / "../1/phase1_chunks_vi.json").resolve())

pipeline = GraphRAGPipeline(config, CHUNKS_JSON)
print("Pipeline ready.")
print(f"  LLM          : {config.llm_provider} / {config.llm_model}")
print(f"  Neo4j db     : {config.neo4j_database}")
print(f"  ColBERT EN   : {len(pipeline.colbert_store._en)} chunks loaded")
print(f"  ColBERT VI   : {len(pipeline.colbert_store._vi)} chunks loaded")


14:28:25 [INFO] graphrag: Loading chunk store from K:\Ds\UCD\CeADAR_Quan\research\1\phase1_chunks_vi.json
14:28:25 [INFO] graphrag: Loaded 39 chunks (68 images, 68 captions across 32 chunks)
14:28:25 [INFO] graphrag: Loading ColBERT store from K:\Ds\UCD\CeADAR_Quan\research\1\colbert-embeddings
14:28:25 [INFO] graphrag: ColBERT store ready — 39 EN embeddings, 39 VI embeddings


Langfuse connected  : http://35.186.40.29:8080


14:28:29 [INFO] graphrag: Embedding dimension validated: 1536d
14:28:29 [INFO] graphrag: GraphRAGPipeline ready (mode=textonly, captions=False, llm=cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit, db=neo4j, colbert_en=39 chunks, verify_math=True)


Pipeline ready.
  LLM          : qwen / cpatonn/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit
  Neo4j db     : neo4j
  ColBERT EN   : 39 chunks loaded
  ColBERT VI   : 39 chunks loaded


In [15]:
result = pipeline.query(
    "What is the recommended seeding rate for wet-seeded rice and how does it affect weed pressure?"
)

print(f"Language : {result.language}")
print(f"Timings  : { {k: round(v,2) for k,v in result.timings.items()} }")
print()
print("=== Answer ===")
print(result.answer)
print()
print("=== Sources ===")
for t in result.traces:
    print(f"  [{t.chunk_id}] {t.title} (pp.{t.start_page}-{t.end_page})")
    print(f"      channels={t.channels}  rrf={t.rrf_score:.4f}")

# ── Langfuse: flush all pending events before the cell finishes ─────────────
if _LANGFUSE_AVAILABLE:
    _get_langfuse_client().flush()


14:28:29 [INFO] graphrag: Query language: en


Loading tokenizer: lightonai/Reason-ModernColBERT ...
Loading backbone ...
Loading Dense projection layer ...


14:28:33 [INFO] graphrag: Retrieval counts — dense: 5, colbert: 10, bm25: 3, graph: 20  (parallel)


  Ready — backbone hidden=768, projection→128, device=cpu



14:28:39 [INFO] graphrag: Pipeline completed in 10.52s


Language : en
Timings  : {'language_detection': 0.0, 'translation': 0.0, 'embedding': 0.18, 'retrieval_parallel': 4.47, 'rrf_fusion': 0.0, 'context_assembly': 0.0, 'generation': 5.87, 'total': 10.52}

=== Answer ===
**Quick Answer:**  
The recommended seeding rate for wet-seeded rice is **no more than 70 kg/ha**, and using this rate—especially with mechanized row or cluster sowing—helps reduce weed pressure by improving light interception and plant competition.

---

**Step-by-Step Guide:**

✅ **Step 1: Use the correct seeding rate**  
- For mechanized sowing (row or cluster), the maximum seed rate should not exceed **70 kg/ha**.  
- If using row sowing, keep it at **≤60 kg/ha** to avoid overcrowding (pp. 26–27, 32).  
- For cluster sowing, seed rate can go up to **120 kg/ha**, but only if using a cluster seeder with proper spacing (pp. 28–30).

✅ **Step 2: Choose the right sowing method to reduce weeds**  
- Use **wide-row–narrow-row sowing (35x15 cm)** or **aerodynamic row sowing (25